# 05 - Train model (ResNet18)

Trains a **ResNet18** classifier on the `data/splits/` set produced by step 04: frozen ImageNet
backbone + a fresh classifier head, class-weighted loss, early stopping. Reports synthetic-test
accuracy, a confusion matrix, and a **zero-shot CLIP** baseline for comparison.

Adapted from Avital's notebook: paths come from `utils.py` (no Colab/Drive); class labels come from
the folders. An optional full fine-tuning pass and an optional real-CCTV evaluation are included.

## Install + imports

In [ ]:
# !pip install torch torchvision scikit-learn transformers matplotlib
import os
# --- choose the GPU here (before torch); change "0" + restart kernel to switch ---
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import numpy as np
import torch, torch.nn as nn
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

import sys
from pathlib import Path
# Make the local shlomi/utils.py importable whether launched from shlomi/ or the repo root.
for _cand in (Path.cwd(), Path.cwd() / "shlomi"):
    if (_cand / "utils.py").exists() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand)); break
import utils

torch.manual_seed(utils.SEED); np.random.seed(utils.SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# --- Experiment tracking (Weights & Biases) ---
# Active ONLY if WANDB_API_KEY is set in the environment (e.g. via shlomi/secrets.env in the
# container), so the notebook runs unchanged with or without tracking. No keys live in the notebook.
import os
USE_WANDB = True
if USE_WANDB:
    import wandb
    wandb.login()                      # reads WANDB_API_KEY from the environment
def wlog(d):
    if USE_WANDB:
        wandb.log(d)
print("W&B logging:", "ON" if USE_WANDB else "off (set WANDB_API_KEY to enable)")

device: cuda
W&B logging: off (set WANDB_API_KEY to enable)


## Paths

In [3]:
train_dir = utils.SPLITS_DIR / "train"
val_dir   = utils.SPLITS_DIR / "val"
test_dir  = utils.SPLITS_DIR / "test"
real_test_dir = utils.DATA_DIR / "real_test"     # optional (created by step 04 if you had real labels)
utils.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## Transforms

Train-time augmentation on top of the baked-in CCTV degradation; eval is just resize+normalize.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(8),
    # transforms.RandomPerspective(distortion_scale=0.2, p=0.4),
    # transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    # transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.2)),
    transforms.ToTensor(),
    # transforms.Lambda(lambda x: x + 0.03 * torch.randn_like(x)),
    transforms.Normalize([0.5]*3, [0.5]*3),
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

## Preview: what `train_transform` does (live per-epoch augmentation)

This is the runtime augmentation applied to each (already-degraded) image **every epoch** - it is NOT
saved to disk. Run this cell to SEE it, then tune the transform above and re-run. (To preview without
running the whole notebook:  `python shlomi/preview_train_transform.py`.)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image as _Image

def _unnorm(t):                       # invert Normalize([0.5]*3,[0.5]*3) -> [0,1] for display
    return (t * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()

_classes = sorted(d.name for d in utils.CLEAN_DIR.iterdir() if d.is_dir())
_K = 5                                 # random augmentations shown per image
_fig, _ax = plt.subplots(len(_classes), _K + 1, figsize=(2 * (_K + 1), 2 * len(_classes)), squeeze=False)
for _r, _cls in enumerate(_classes):
    _f = sorted((utils.CLEAN_DIR / _cls).glob("*"))[0]
    _img = _Image.open(_f).convert("RGB")
    _ax[_r][0].imshow(_img.resize((224, 224))); _ax[_r][0].axis("off")
    _ax[_r][0].set_title(f"{_cls} (original)", fontsize=9)
    for _k in range(_K):
        _ax[_r][_k + 1].imshow(_unnorm(train_transform(_img))); _ax[_r][_k + 1].axis("off")
        if _r == 0: _ax[_r][_k + 1].set_title(f"aug {_k + 1}", fontsize=9)
_fig.suptitle("train_transform preview - live per-epoch augmentation (NOT saved to disk)", fontsize=11)
plt.tight_layout()
utils.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(utils.RESULTS_DIR / "train_transform_preview.png", dpi=110, bbox_inches="tight")
plt.show()

## Datasets + loaders

In [ ]:
train_dataset = datasets.ImageFolder(str(train_dir), transform=train_transform)
val_dataset   = datasets.ImageFolder(str(val_dir),   transform=test_transform)
test_dataset  = datasets.ImageFolder(str(test_dir),  transform=test_transform)
CLASSES = train_dataset.classes
print("classes:", CLASSES)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=4)

## Model: ResNet18, frozen backbone + new head

In [ ]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
for p in model.parameters():
    p.requires_grad = False                      # feature-extraction baseline
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
model = model.to(device)

## Loss (class-weighted) + optimizer

In [ ]:
from collections import Counter
counts = Counter(lbl for _, lbl in train_dataset.samples)
total = sum(counts.values())
weights = torch.tensor([total / counts[i] for i in range(len(CLASSES))], dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

## Train / eval helpers

In [ ]:
def run_epoch(model, loader, train, optimizer=None):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    torch.set_grad_enabled(train)
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if train: optimizer.zero_grad()
        out = model(x); loss = criterion(out, y)
        if train: loss.backward(); optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item(); total += y.size(0)
    torch.set_grad_enabled(True)
    return total_loss / total, correct / total

## Train (frozen backbone) with early stopping

In [ ]:
EPOCHS, PATIENCE = 30, 7
best_val, counter = 0.0, 0
hist = {"tr_loss": [], "tr_acc": [], "va_loss": [], "va_acc": []}
ckpt = utils.RESULTS_DIR / "resnet18_frozen_best.pth"
if USE_WANDB:
    wandb.init(project=os.environ.get("WANDB_PROJECT", "plate-classification"),
               name=os.environ.get("WANDB_RUN_NAME", "shlomi_resnet18_frozen"),
               config={"epochs": EPOCHS, "patience": PATIENCE, "lr": 1e-4,
                       "model": "resnet18_frozen", "classes": CLASSES})

for epoch in range(EPOCHS):
    tr_loss, tr_acc = run_epoch(model, train_loader, True, optimizer)
    va_loss, va_acc = run_epoch(model, val_loader, False)
    for k, v in zip(hist, (tr_loss, tr_acc, va_loss, va_acc)): hist[k].append(v)
    print(f"epoch {epoch+1:02d} | train {tr_acc:.3f}/{tr_loss:.3f} | val {va_acc:.3f}/{va_loss:.3f}")
    wlog({"train_loss": tr_loss, "train_acc": tr_acc, "val_loss": va_loss, "val_acc": va_acc, "epoch": epoch + 1})
    if va_acc > best_val:
        best_val, counter = va_acc, 0
        torch.save(model.state_dict(), ckpt); print("  saved best")
    else:
        counter += 1
        if counter >= PATIENCE:
            print("  early stopping"); break
model.load_state_dict(torch.load(ckpt)); print("loaded best (val acc %.3f)" % best_val)

## Training curves

In [ ]:
ep = range(1, len(hist["tr_loss"]) + 1)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.plot(ep, hist["tr_loss"], label="train"); plt.plot(ep, hist["va_loss"], label="val")
plt.title("Loss"); plt.xlabel("epoch"); plt.legend()
plt.subplot(1, 2, 2); plt.plot(ep, hist["tr_acc"], label="train"); plt.plot(ep, hist["va_acc"], label="val")
plt.title("Accuracy"); plt.xlabel("epoch"); plt.legend(); plt.show()

## Synthetic test + confusion matrix + binary decision accuracy

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score

def predict(model, loader):
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            ps.extend(model(x.to(device)).argmax(1).cpu().numpy()); ys.extend(y.numpy())
    return np.array(ys), np.array(ps)

test_loss, test_acc = run_epoch(model, test_loader, False)
print(f"Synthetic test: acc {test_acc:.3f} | loss {test_loss:.3f}")

y_true, y_pred = predict(model, test_loader)
print("macro-F1:", round(f1_score(y_true, y_pred, average='macro'), 3))
wlog({"test_acc": test_acc, "test_loss": test_loss, "macro_f1": float(f1_score(y_true, y_pred, average="macro"))})
if USE_WANDB:
    wandb.log({"confusion_matrix": wandb.plot.confusion_matrix(probs=None, y_true=y_true, preds=y_pred, class_names=CLASSES)})
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASSES).plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion matrix - synthetic test"); plt.tight_layout(); plt.show()

# Binary clear / do-not-clear accuracy (unclassified -> abstain, reported separately).
to_bin = {i: utils.to_binary(c) for i, c in enumerate(CLASSES)}
mask = np.array([to_bin[t] != "uncertain" for t in y_true])
if mask.any():
    bt = [to_bin[t] for t in y_true[mask]]; bp = [to_bin[p] for p in y_pred[mask]]
    print("Binary (clear/do_not_clear) acc:", round(np.mean(np.array(bt) == np.array(bp)), 3))

## Zero-shot CLIP baseline

In [ ]:
from transformers import CLIPModel, CLIPProcessor
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", use_safetensors=True).to(device)
clip_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

CLASS_DESC = {
    "clean": "a pristine clean empty plate with no food",
    "empty": "a used empty plate with crumbs, smears and food residue but no food",
    "finished": "a used plate after a meal, with only leftover food scraps, crumbs and sauce",
    "finished_leftovers": "a plate with a small amount of leftover food scraps",
    "full": "a full plate with a complete serving of food",
    "unclassified": "a heavily blurred, noisy, unreadable low-quality photo of a plate",
}
labels_text = [CLASS_DESC.get(c, f"a plate, category {c}") for c in CLASSES]

def clip_eval(loader):
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            for i in range(len(x)):
                img = transforms.ToPILImage()(x[i] * 0.5 + 0.5)   # un-normalize for CLIP
                inp = clip_proc(text=labels_text, images=img, return_tensors="pt", padding=True).to(device)
                pred = clip_model(**inp).logits_per_image.softmax(1).argmax().item()
                correct += int(pred == y[i].item()); total += 1
    return correct / total

clip_acc = round(clip_eval(test_loader), 3)
print("CLIP zero-shot (synthetic test):", clip_acc)
wlog({"clip_acc": clip_acc})

## (Optional) Full fine-tuning

Unfreeze the whole backbone and train at a low LR - report this vs the frozen baseline (the rubric
asks for fine-tuned vs off-the-shelf).

In [ ]:
RUN_FINETUNE = True    # set True to also fine-tune the whole backbone (usually higher accuracy)
if RUN_FINETUNE:
    for p in model.parameters(): p.requires_grad = True          # unfreeze everything
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)
    best_val, counter = 0.0, 0
    ft_hist = {"tr_loss": [], "tr_acc": [], "va_loss": [], "va_acc": []}
    ckpt_ft = utils.RESULTS_DIR / "resnet18_finetuned_best.pth"

    if USE_WANDB:                                   # log FT as its OWN W&B run (same metric names as LP -> overlay)
        wandb.finish()                              # close the linear-probe run first
        wandb.init(project=os.environ.get("WANDB_PROJECT", "plate-classification"),
                   name="shlomi_resnet18_finetuned",
                   config={"epochs": 15, "patience": PATIENCE, "lr": 1e-5,
                           "model": "resnet18_finetuned", "classes": CLASSES})
    for epoch in range(15):
        tr_loss, tr_acc = run_epoch(model, train_loader, True, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader, False)
        for k, v in zip(ft_hist, (tr_loss, tr_acc, va_loss, va_acc)): ft_hist[k].append(v)
        print(f"[FT] epoch {epoch+1:02d} | train {tr_acc:.3f}/{tr_loss:.3f} | val {va_acc:.3f}/{va_loss:.3f}")
        wlog({"train_loss": tr_loss, "train_acc": tr_acc, "val_loss": va_loss, "val_acc": va_acc, "epoch": epoch + 1})
        if va_acc > best_val:
            best_val, counter = va_acc, 0
            torch.save(model.state_dict(), ckpt_ft); print("  saved best (ft)")
        else:
            counter += 1
            if counter >= PATIENCE:
                print("  early stopping (ft)"); break
    model.load_state_dict(torch.load(ckpt_ft)); print("loaded best ft (val acc %.3f)" % best_val)
else:
    print("RUN_FINETUNE is False - skipping fine-tuning.")

## Fine-tuned model report (curves + confusion matrix + metrics)

In [ ]:
# Fine-tuned model: curves + confusion matrix + metrics (only if FT ran). Mirrors the linear-probe report.
if RUN_FINETUNE:
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score

    # --- training curves ---
    ep = range(1, len(ft_hist["tr_loss"]) + 1)
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1); plt.plot(ep, ft_hist["tr_loss"], label="train"); plt.plot(ep, ft_hist["va_loss"], label="val")
    plt.title("FT Loss"); plt.xlabel("epoch"); plt.legend()
    plt.subplot(1, 2, 2); plt.plot(ep, ft_hist["tr_acc"], label="train"); plt.plot(ep, ft_hist["va_acc"], label="val")
    plt.title("FT Accuracy"); plt.xlabel("epoch"); plt.legend(); plt.show()

    # --- synthetic test + confusion matrix + macro-F1 ---
    ft_test_loss, ft_test_acc = run_epoch(model, test_loader, False)
    print(f"[FT] Synthetic test: acc {ft_test_acc:.3f} | loss {ft_test_loss:.3f}")
    y_true, y_pred = predict(model, test_loader)
    ft_f1 = f1_score(y_true, y_pred, average="macro")
    print("[FT] macro-F1:", round(ft_f1, 3))
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASSES).plot(cmap="Blues", xticks_rotation=45)
    plt.title("Confusion matrix - synthetic test (fine-tuned)"); plt.tight_layout(); plt.show()

    # --- binary clear / do-not-clear accuracy ---
    to_bin = {i: utils.to_binary(c) for i, c in enumerate(CLASSES)}
    mask = np.array([to_bin[t] != "uncertain" for t in y_true])
    ft_bin = None
    if mask.any():
        bt = [to_bin[t] for t in y_true[mask]]; bp = [to_bin[p] for p in y_pred[mask]]
        ft_bin = float(np.mean(np.array(bt) == np.array(bp)))
        print("[FT] Binary (clear/do_not_clear) acc:", round(ft_bin, 3))

    # --- W&B (ft_-labelled, so LP vs FT are comparable in one run) ---
    wlog({"test_acc": ft_test_acc, "test_loss": ft_test_loss, "macro_f1": float(ft_f1),
          **({"binary_acc": ft_bin} if ft_bin is not None else {})})
    if USE_WANDB:
        wandb.log({"confusion_matrix": wandb.plot.confusion_matrix(probs=None, y_true=y_true, preds=y_pred, class_names=CLASSES)})
else:
    print("RUN_FINETUNE is False - no FT report.")

## (Optional) Real-CCTV evaluation

Only runs if step 04 produced a labelled `data/real_test/`. Per CLAUDE.md the real photos are
calibration/inspiration - this is a *bonus sanity check*, not a primary metric.

In [1]:
# --- Real-CCTV evaluation (STANDALONE: runs on its own, even after a kernel restart) ---
# Loads the trained model from results/, so you do NOT need to re-run the whole notebook.
import sys, numpy as np, torch
from pathlib import Path
from torchvision import datasets, transforms
from torchvision.models import resnet18
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
for _c in (Path.cwd(), Path.cwd() / "shlomi"):
    if (_c / "utils.py").exists() and str(_c) not in sys.path:
        sys.path.insert(0, str(_c)); break
import utils

device = "cuda" if torch.cuda.is_available() else "cpu"
real_test_dir = utils.DATA_DIR / "real_test"

if not (real_test_dir.is_dir() and any(real_test_dir.iterdir())):
    print("No labelled real_test/ at", real_test_dir, "- nothing to evaluate.")
else:
    tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                             transforms.Normalize([0.5] * 3, [0.5] * 3)])
    real_ds = datasets.ImageFolder(str(real_test_dir), transform=tf)
    real_loader = DataLoader(real_ds, batch_size=64, shuffle=False)

    # load the best saved model (prefer fine-tuned, else frozen)
    ckpt = utils.RESULTS_DIR / "resnet18_finetuned_best.pth"
    if not ckpt.exists():
        ckpt = utils.RESULTS_DIR / "resnet18_frozen_best.pth"
    sd = torch.load(ckpt, map_location=device)
    model = resnet18(weights=None)
    model.fc = torch.nn.Linear(model.fc.in_features, sd["fc.weight"].shape[0])
    model.load_state_dict(sd); model.to(device).eval()
    print("loaded", ckpt.name, "| real classes:", real_ds.classes)

    ys, ps = [], []
    with torch.no_grad():
        for x, y in real_loader:
            ps.extend(model(x.to(device)).argmax(1).cpu().numpy()); ys.extend(y.numpy())
    ys, ps = np.array(ys), np.array(ps)
    print("Real test acc:", round(float((ys == ps).mean()), 3))
    ConfusionMatrixDisplay(confusion_matrix(ys, ps), display_labels=real_ds.classes).plot(cmap="Blues", xticks_rotation=45)
    plt.title("Confusion matrix - real test"); plt.tight_layout(); plt.show()

NameError: name 'real_test_dir' is not defined

In [ ]:
# Close the W&B run (if tracking was on)
if USE_WANDB:
    wandb.finish()